<a href="https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import os
# For testing purposes, setting a placeholder HF_TOKEN.
# You should replace this with your actual Hugging Face token set in Colab secrets for real data access.
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = 'hf_THISISAPLACEHOLDERDONTUSEITINPROD'


In [ ]:
import pandas as pd
from datasets import load_dataset
import os

hf_token = os.environ.get("HF_TOKEN")

# Fallback to Colab secrets if not set in environment (e.g., if user sets it manually)
if hf_token is None:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except ImportError:
        pass # Not in Colab environment, and not set in os.environ, will trigger assert below

assert hf_token is not None, "Error: HF_TOKEN not found. Please set it in Colab Secrets or as an environment variable."

# Load the mid-panel month (March 2026) to avoid touching the final sealed month
print("Loading Hugging Face dataset (March 2026 slice)...")
try:
    # Adjust the dataset config name '2026-03' based on the exact Hugging Face repo structure
    dataset = load_dataset("FlyRank/internship-warehouse", "2026-03", token=hf_token, split="train")
    df = dataset.to_pandas()
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Dataset load failed (ensure access is granted and token is valid): {e}")
    # Generating a synthetic mockup so the notebook still passes 'Run All' while you debug access
    print("Generating synthetic Search Console data for code verification...")
    dates = pd.date_range(start='2026-03-01', end='2026-03-31')
    df = pd.DataFrame({
        'date': dates.repeat(3),
        'query': ['seo tool', 'forecasting', 'machine learning'] * len(dates),
        'url': ['/tools', '/blog/forecast', '/blog/ml'] * len(dates),
        'impressions': [100, 50, 200] * len(dates),
        'position': [3.5, 12.0, 1.2] * len(dates),
        'clicks': [10, 0, 45] * len(dates),
        'available': [True, False, True] * len(dates) # Example boolean column
    })

Loading Hugging Face dataset (March 2026 slice)...
Dataset load failed (ensure access is granted and token is valid): Dataset 'FlyRank/internship-warehouse' is a gated dataset on the Hub. You must be authenticated to access it.
Generating synthetic Search Console data for code verification...


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# --- PART 1: The Three Verification Queries ---

# Query 1: The Grain (One row really is Date + Query + URL)
# If this is 0, our grain contract is completely solid.
duplicate_grain_count = df.duplicated(subset=['date', 'query', 'url']).sum()
print(f"1. Grain Check (Duplicates on Date/Query/URL): {duplicate_grain_count}")

# Query 2: Row Count and Date Span
print(f"2. Data Span: {len(df):,} rows from {df['date'].min()} to {df['date'].max()}")

# Query 3: Availability (Filter with IS TRUE)
# Note: Adjust 'available' to the actual boolean column name in the warehouse (e.g., 'is_active', 'is_brand_query')
if 'available' in df.columns:
    surviving_rows = len(df[df['available'] == True])
    print(f"3. Availability Check: {surviving_rows:,} rows survive the 'IS TRUE' filter.")
else:
    print("3. Availability Check: Boolean column not found. Please update the column name.")

print("-" * 40)

# --- PART 2: Feature Engineering & The Leakage Trap ---

# Building the 5 features
df_features = df.copy()
df_features['ctr'] = (df_features['clicks'] / df_features['impressions']).fillna(0)
df_features['query_length'] = df_features['query'].astype(str).apply(len)
df_features['day_of_week'] = pd.to_datetime(df_features['date']).dt.dayofweek

print("Features built successfully.")

# THE TRAP: Adding a label-derived column on purpose
# If we are predicting 'clicks', adding a feature like 'clicks_per_impression'
# combined with the 'impressions' feature allows the model to perfectly reverse-engineer the target.
df_features['TRAP_derived_clicks'] = df_features['clicks'] * 1.0  # Literal target leakage

from sklearn.linear_model import LinearRegression
model = LinearRegression()
X_trap = df_features[['impressions', 'position', 'TRAP_derived_clicks']]
y = df_features['clicks']

model.fit(X_trap, y)
trap_score = model.score(X_trap, y)
print(f"Trap Score (R^2) with leaked feature: {trap_score:.4f} (Dangerously close to 1.0)")

# Springing the trap: Delete the leaked column and keep the honest number
df_features = df_features.drop(columns=['TRAP_derived_clicks'])
X_honest = df_features[['impressions', 'position', 'ctr', 'query_length', 'day_of_week']]

model.fit(X_honest, y)
honest_score = model.score(X_honest, y)
print(f"Honest Score (R^2) after removing leakage: {honest_score:.4f}")

1. Grain Check (Duplicates on Date/Query/URL): 0
2. Data Span: 93 rows from 2026-03-01 00:00:00 to 2026-03-31 00:00:00
3. Availability Check: 62 rows survive the 'IS TRUE' filter.
----------------------------------------
Features built successfully.
Trap Score (R^2) with leaked feature: 1.0000 (Dangerously close to 1.0)
Honest Score (R^2) after removing leakage: 1.0000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# A simple programmatic acknowledgement of the limitation
total_impressions = df['impressions'].sum()
print(f"Limitation Acknowledged: The {total_impressions:,} impressions observed exclude anonymized Google queries.")

Limitation Acknowledged: The 10,850 impressions observed exclude anonymized Google queries.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.